In [ ]:
import os
import glob
import re
import json
import time
import pickle
import gc
import warnings
import numpy as np
import pandas as pd
from datetime import datetime
from scipy.io import loadmat
from scipy.special import logsumexp
import scipy.linalg
from sklearn.mixture import GaussianMixture

warnings.filterwarnings('ignore')

# ---------------------------------------------------------------------------
# Numba JIT kernels for forward and backward passes.
# ---------------------------------------------------------------------------
try:
    from numba import njit

    @njit(cache=True)
    def _forward_numba(log_pi, log_A, log_em):
        T, K       = log_em.shape
        log_alpha  = np.empty((T, K))
        log_scales = np.empty(T)
        for j in range(K):
            log_alpha[0, j] = log_pi[j] + log_em[0, j]
        ls = log_alpha[0, 0]
        for j in range(1, K):
            if log_alpha[0, j] > ls:
                ls = log_alpha[0, j]
        s = 0.0
        for j in range(K):
            s += np.exp(log_alpha[0, j] - ls)
        log_scales[0] = ls + np.log(s)
        for j in range(K):
            log_alpha[0, j] -= log_scales[0]
        for t in range(1, T):
            for j in range(K):
                mx = log_alpha[t-1, 0] + log_A[0, j]
                for i in range(1, K):
                    v = log_alpha[t-1, i] + log_A[i, j]
                    if v > mx:
                        mx = v
                acc = 0.0
                for i in range(K):
                    acc += np.exp(log_alpha[t-1, i] + log_A[i, j] - mx)
                log_alpha[t, j] = mx + np.log(acc) + log_em[t, j]
            mx = log_alpha[t, 0]
            for j in range(1, K):
                if log_alpha[t, j] > mx:
                    mx = log_alpha[t, j]
            s = 0.0
            for j in range(K):
                s += np.exp(log_alpha[t, j] - mx)
            log_scales[t] = mx + np.log(s)
            for j in range(K):
                log_alpha[t, j] -= log_scales[t]
        return log_alpha, log_scales

    @njit(cache=True)
    def _backward_numba(log_A, log_em, log_scales):
        T, K     = log_em.shape
        log_beta = np.zeros((T, K))
        for i in range(K):
            log_beta[T-1, i] = -log_scales[T-1]
        for t in range(T - 2, -1, -1):
            for i in range(K):
                mx = log_A[i, 0] + log_em[t+1, 0] + log_beta[t+1, 0]
                for j in range(1, K):
                    v = log_A[i, j] + log_em[t+1, j] + log_beta[t+1, j]
                    if v > mx:
                        mx = v
                acc = 0.0
                for j in range(K):
                    acc += np.exp(log_A[i, j] + log_em[t+1, j] + log_beta[t+1, j] - mx)
                log_beta[t, i] = mx + np.log(acc) - log_scales[t]
        return log_beta

    _NUMBA_AVAILABLE = True
    print("Numba available — forward/backward will use JIT kernels.")

except ImportError:
    _NUMBA_AVAILABLE = False
    print("Numba not found — falling back to NumPy loops. "
          "Install with:  pip install numba --quiet")


# =============================================================================
# Time manager
# =============================================================================

class TimeManager:
    def __init__(self, max_hours: float = 11.5):
        self.start_time  = time.time()
        self.max_seconds = max_hours * 3600
        self._last_ckpt  = self.start_time

    def elapsed_seconds(self) -> float:
        return time.time() - self.start_time

    def elapsed_hours(self) -> float:
        return self.elapsed_seconds() / 3600

    def remaining_hours(self) -> float:
        return max(0.0, self.max_seconds / 3600 - self.elapsed_hours())

    def format_elapsed(self) -> str:
        s      = int(self.elapsed_seconds())
        h, rem = divmod(s, 3600)
        m, sec = divmod(rem, 60)
        return f"{h:02d}:{m:02d}:{sec:02d}"

    def should_stop(self, buffer_minutes: float = 30.0) -> bool:
        return (self.elapsed_seconds() + buffer_minutes * 60) >= self.max_seconds

    def estimate_remaining(self, completed: int, total: int) -> int:
        if completed == 0:
            return total
        avg_sec = self.elapsed_seconds() / completed
        avail   = self.remaining_hours() * 3600 - 1800
        return max(0, int(avail / avg_sec))

    def print_status(self, completed: int, total: int):
        print(f"  [time]  elapsed={self.format_elapsed()}  "
              f"remaining={self.remaining_hours():.2f}h  "
              f"progress={completed}/{total}  "
              f"est_can_finish~{self.estimate_remaining(completed, total)}")


# =============================================================================
# Reproducible seed
# =============================================================================

def _make_seed(subject_type: str, subject_id: str) -> int:
    tag = f"{subject_type}_{subject_id}"
    h   = 2166136261
    for ch in tag.encode():
        h ^= ch
        h  = (h * 16777619) & 0xFFFFFFFF
    return 42 + (h % 100_000)


# =============================================================================
# Data loading
# =============================================================================

def load_preprocessed_mat(filepath: str) -> tuple:
    """
    Load a preprocessed .mat file produced by EEGPreprocessing.

    Returns
    -------
    signal : np.ndarray, shape (n_samples, n_channels)
        EEG signal in time-major layout expected by the HMM.
    mat_label : str or None
        'label' field stored inside the .mat file ('Control' or 'ADHD'),
        used as a cross-check against the directory-derived subject_type.
    mat_subject_id : str or None
        'subject_id' field stored inside the .mat file.
    """
    mat = loadmat(filepath)
    if 'signal' not in mat:
        raise KeyError(
            f"Key 'signal' not found in {filepath}. "
            f"Available keys: {[k for k in mat if not k.startswith('__')]}"
        )
    signal = mat['signal'].astype(np.float64)
    if signal.ndim == 1:
        signal = signal[np.newaxis, :]
    # EEGPreprocessing always saves signal as [n_channels, n_samples].
    # HMM expects [n_samples, n_channels] (time-major), so always transpose.
    # Shape check is a safety assertion only — crash loudly if unexpected.
    if signal.ndim != 2:
        raise ValueError(
            f"Expected 2-D signal array, got shape {signal.shape} in {filepath}"
        )
    n_channels_expected_max = 19   # full 10-20 cap; frontal subset is 7
    if signal.shape[0] <= n_channels_expected_max:
        # Confirmed channels-first layout: (n_channels, n_samples)
        signal = signal.T          # -> (n_samples, n_channels)
    elif signal.shape[1] <= n_channels_expected_max:
        # Already time-major layout: (n_samples, n_channels) — keep as-is
        pass
    else:
        raise ValueError(
            f"Cannot determine signal orientation: shape={signal.shape} "
            f"in {filepath}. Neither axis has <= {n_channels_expected_max} "
            f"elements to identify the channel axis."
        )

    # Read embedded metadata for cross-checking.
    def _extract_str(key: str):
        val = mat.get(key, None)
        if val is None:
            return None
        val = np.asarray(val).ravel()
        if val.dtype.kind in ('U', 'S', 'O'):
            s = str(val[0])
            return s.strip()
        return None

    mat_label      = _extract_str('label')
    mat_subject_id = _extract_str('subject_id')
    return signal, mat_label, mat_subject_id


def parse_preprocessed_filename(fname: str):
    """
    Extract subject_id from a preprocessed filename.

    Expected format (produced by EEGPreprocessing):
        {subject_id}_preprocessed.mat

    where subject_id is the raw token from the original file, e.g.:
        v107_preprocessed.mat   -> subject_id = 'v107'   (ADHD)
        v41p_preprocessed.mat   -> subject_id = 'v41p'   (Control, 'p' suffix)

    Subject type is NOT encoded in the filename; it is derived from the
    directory the file lives in (Control_prep or ADHD_prep).

    Returns subject_id string, or None if the filename does not match.
    """
    base    = os.path.splitext(os.path.basename(fname))[0]
    pattern = re.compile(r'^(.+)_preprocessed$', re.IGNORECASE)
    m       = pattern.match(base.strip())
    if m is None:
        return None
    return m.group(1)


def _subject_type_from_dir(directory: str) -> str:
    """
    Infer subject type label from the name of the directory.

    Control_prep -> 'CONTROL'
    ADHD_prep    -> 'ADHD'
    """
    base = os.path.basename(os.path.normpath(directory)).upper()
    if 'CONTROL' in base:
        return 'CONTROL'
    if 'ADHD' in base:
        return 'ADHD'
    raise ValueError(
        f"Cannot infer subject type from directory name: '{directory}'. "
        f"Directory must contain 'Control' or 'ADHD'."
    )


def discover_files(control_prep_dir: str, adhd_prep_dir: str) -> list:
    """
    Discover all preprocessed .mat files in both class directories.

    Returns a list of (filepath, subject_type, subject_id) tuples.
    subject_type is derived from the directory ('CONTROL' or 'ADHD').
    subject_id   is the raw token before '_preprocessed' in the filename.
    """
    records = []
    for directory in [control_prep_dir, adhd_prep_dir]:
        if not os.path.isdir(directory):
            print(f"  [WARNING] Directory not found: {directory}")
            continue
        subject_type = _subject_type_from_dir(directory)
        for fp in sorted(glob.glob(os.path.join(directory, "*.mat"))):
            subject_id = parse_preprocessed_filename(fp)
            if subject_id is None:
                print(f"  [SKIP] Unrecognised filename: {os.path.basename(fp)}")
                continue
            records.append((fp, subject_type, subject_id))
    return records


# =============================================================================
# Configuration helpers
# =============================================================================

def _config_tag(n_states: int, n_gmm: int) -> str:
    return f"N{n_states}G{n_gmm}"


def _max_iter_for_states(n_states: int) -> int:
    """Iteration budget scales with model complexity."""
    if n_states <= 2:
        return 50
    elif n_states <= 4:
        return 100
    elif n_states == 5:
        return 150
    else:
        return 200


def _config_output_dir(base_dir: str, n_states: int, n_gmm: int) -> str:
    return os.path.join(base_dir, _config_tag(n_states, n_gmm))


def resolve_configs(configs_to_run) -> list:
    """
    Resolve CONFIGS_TO_RUN to a list of (n_states, n_gmm) tuples.

    Accepts:
        'all'                       -> all 40 configurations (states 1-8, gmm 1-5)
        [(3, 3), (5, 2), ...]       -> explicit list of (n_states, n_gmm) tuples
    """
    if configs_to_run == 'all':
        return [(s, g) for s in range(1, 9) for g in range(1, 6)]
    if isinstance(configs_to_run, list):
        validated = []
        for entry in configs_to_run:
            if (not isinstance(entry, tuple) or len(entry) != 2
                    or not all(isinstance(x, int) for x in entry)):
                raise ValueError(
                    f"Each entry in CONFIGS_TO_RUN must be an (int, int) tuple. "
                    f"Got: {entry!r}"
                )
            n_states, n_gmm = entry
            if not (1 <= n_states <= 8):
                raise ValueError(f"n_states must be in [1, 8]. Got: {n_states}")
            if not (1 <= n_gmm <= 5):
                raise ValueError(f"n_gmm must be in [1, 5]. Got: {n_gmm}")
            validated.append((n_states, n_gmm))
        return validated
    raise ValueError(
        "CONFIGS_TO_RUN must be 'all' or a list of (n_states, n_gmm) tuples."
    )


def discover_existing_configs(base_dir: str) -> set:
    """
    Scan base_dir for subdirectories matching the NxGy pattern that contain
    at least one completed checkpoint (progress.json with completed entries).
    Returns a set of (n_states, n_gmm) tuples.
    """
    found = set()
    pattern = re.compile(r'^N(\d+)G(\d+)$')
    if not os.path.isdir(base_dir):
        return found
    for entry in os.listdir(base_dir):
        m = pattern.match(entry)
        if m is None:
            continue
        prog_path = os.path.join(base_dir, entry, 'progress.json')
        if not os.path.exists(prog_path):
            continue
        try:
            with open(prog_path) as f:
                prog = json.load(f)
            if prog.get('completed'):
                found.add((int(m.group(1)), int(m.group(2))))
        except Exception:
            pass
    return found


# =============================================================================
# Multivariate GMM-HMM
# =============================================================================

class MultivariateGaussianMixtureHMM:

    def __init__(
        self,
        n_states:         int   = 3,
        n_gmm_components: int   = 3,
        max_iter:         int   = 100,
        tol:              float = 1.0,
        tol_scale_ll:     float = 1.0,
        tol_scale_pi:     float = 1e-3,
        random_state:     int   = 42,
    ):
        self.n_states         = n_states
        self.n_gmm_components = n_gmm_components
        self.max_iter         = max_iter
        self.tol              = tol
        self.tol_ll           = tol * tol_scale_ll
        self.tol_pi           = tol * tol_scale_pi
        self.random_state     = random_state

        self.pi          = None
        self.A           = None
        self.gmm_weights = None
        self.gmm_means   = None
        self.gmm_covars  = None
        self._chol       = None
        self._log_det    = None

        self.train_log_likelihoods = []
        self.val_log_likelihoods   = []
        self.convergence_flags     = []

        self.best_val_ll       = -np.inf
        self.best_iteration    = 0
        self._best_params_snap = None

        self.converged             = False
        self.final_iteration       = 0
        self.convergence_criterion = 'max_iterations'
        self.n_features            = None
        self.stationary_pi         = None

    # ------------------------------------------------------------------
    def _update_cholesky(self):
        K, M, D = self.n_states, self.n_gmm_components, self.n_features
        self._chol    = np.zeros((K, M, D, D))
        self._log_det = np.zeros((K, M))
        eye = np.eye(D) * 1e-6
        for s in range(K):
            for k in range(M):
                cov = self.gmm_covars[s, k] + eye
                try:
                    L = np.linalg.cholesky(cov)
                except np.linalg.LinAlgError:
                    L = np.diag(np.sqrt(np.maximum(np.diag(cov), 1e-10)))
                self._chol[s, k]    = L
                self._log_det[s, k] = 2.0 * np.sum(np.log(np.diag(L)))

    # ------------------------------------------------------------------
    def _initialize_parameters(self, X: np.ndarray):
        np.random.seed(self.random_state)
        T, D            = X.shape
        self.n_features = D
        K, M            = self.n_states, self.n_gmm_components

        self.pi          = np.ones(K) / K
        self.A           = np.random.dirichlet(np.ones(K), K)
        self.gmm_weights = np.zeros((K, M))
        self.gmm_means   = np.zeros((K, M, D))
        self.gmm_covars  = np.zeros((K, M, D, D))

        seg = max(1, T // K)
        for s in range(K):
            sl  = slice(s * seg, (s + 1) * seg if s < K - 1 else T)
            dat = X[sl]
            if len(dat) < M * 2:
                self.gmm_weights[s] = np.ones(M) / M
                mu  = dat.mean(axis=0)
                cov = (np.cov(dat.T) if dat.shape[0] > 1
                       else np.eye(D)) + np.eye(D) * 1e-6
                for k in range(M):
                    self.gmm_means[s, k]  = mu + np.random.randn(D) * 0.01
                    self.gmm_covars[s, k] = cov
            else:
                try:
                    g = GaussianMixture(
                        n_components=M,
                        random_state=self.random_state + s,
                        covariance_type='full',
                        reg_covar=1e-6,
                    ).fit(dat)
                    self.gmm_weights[s] = g.weights_
                    self.gmm_means[s]   = g.means_
                    self.gmm_covars[s]  = g.covariances_
                except Exception:
                    self.gmm_weights[s] = np.ones(M) / M
                    mu  = dat.mean(axis=0)
                    cov = np.cov(dat.T) + np.eye(D) * 1e-6
                    for k in range(M):
                        self.gmm_means[s, k]  = mu
                        self.gmm_covars[s, k] = cov
            for k in range(M):
                self.gmm_covars[s, k] += np.eye(D) * 1e-6

        self._update_cholesky()

    # ------------------------------------------------------------------
    def _log_emission(self, X: np.ndarray) -> np.ndarray:
        T, D      = X.shape
        K, M      = self.n_states, self.n_gmm_components
        log_const = -0.5 * D * np.log(2.0 * np.pi)
        log_comp  = np.empty((K, M, T))
        for s in range(K):
            for k in range(M):
                diff          = X - self.gmm_means[s, k]
                z             = scipy.linalg.solve_triangular(
                    self._chol[s, k], diff.T, lower=True).T
                maha          = np.sum(z ** 2, axis=1)
                log_comp[s,k] = (np.log(self.gmm_weights[s,k] + 1e-12)
                                 + log_const
                                 - 0.5 * self._log_det[s,k]
                                 - 0.5 * maha)
        return logsumexp(log_comp, axis=1).T

    # ------------------------------------------------------------------
    def _forward(self, log_em: np.ndarray):
        log_A  = np.log(self.A  + 1e-12)
        log_pi = np.log(self.pi + 1e-12)
        if _NUMBA_AVAILABLE:
            return _forward_numba(log_pi, log_A, log_em)
        T, K       = log_em.shape
        log_alpha  = np.empty((T, K))
        log_scales = np.empty(T)
        log_alpha[0]  = log_pi + log_em[0]
        log_scales[0] = logsumexp(log_alpha[0])
        log_alpha[0] -= log_scales[0]
        for t in range(1, T):
            acc           = logsumexp(log_alpha[t-1, :, None] + log_A, axis=0)
            log_alpha[t]  = acc + log_em[t]
            log_scales[t] = logsumexp(log_alpha[t])
            log_alpha[t] -= log_scales[t]
        return log_alpha, log_scales

    # ------------------------------------------------------------------
    def _backward(self, log_em: np.ndarray, log_scales: np.ndarray):
        log_A = np.log(self.A + 1e-12)
        if _NUMBA_AVAILABLE:
            return _backward_numba(log_A, log_em, log_scales)
        T, K      = log_em.shape
        log_beta  = np.zeros((T, K))
        log_beta[T-1] = -log_scales[T-1]
        for t in range(T - 2, -1, -1):
            acc         = logsumexp(log_A + log_em[t+1] + log_beta[t+1], axis=1)
            log_beta[t] = acc - log_scales[t]
        return log_beta

    # ------------------------------------------------------------------
    def _e_step(self, log_alpha, log_beta, log_em):
        T, K  = log_alpha.shape
        log_A = np.log(self.A + 1e-12)
        gamma = np.exp(log_alpha + log_beta)
        gamma /= gamma.sum(axis=1, keepdims=True) + 1e-12
        log_xi = (log_alpha[:-1, :, None]
                  + log_A[None, :, :]
                  + log_em[1:, None, :]
                  + log_beta[1:, None, :])
        xi      = np.exp(log_xi)
        xi     /= xi.sum(axis=(1, 2), keepdims=True) + 1e-12
        return gamma, xi

    # ------------------------------------------------------------------
    def _gmm_posteriors(self, X: np.ndarray, gamma: np.ndarray) -> np.ndarray:
        T, D      = X.shape
        K, M      = self.n_states, self.n_gmm_components
        log_const = -0.5 * D * np.log(2.0 * np.pi)
        log_comp  = np.empty((K, M, T))
        for s in range(K):
            for k in range(M):
                diff          = X - self.gmm_means[s, k]
                z             = scipy.linalg.solve_triangular(
                    self._chol[s, k], diff.T, lower=True).T
                maha          = np.sum(z ** 2, axis=1)
                log_comp[s,k] = (np.log(self.gmm_weights[s,k] + 1e-12)
                                 + log_const
                                 - 0.5 * self._log_det[s,k]
                                 - 0.5 * maha)
        log_comp_tks = log_comp.transpose(2, 0, 1)
        return np.exp(
            log_comp_tks - logsumexp(log_comp_tks, axis=2, keepdims=True)
        )

    # ------------------------------------------------------------------
    def _m_step(self, X: np.ndarray, gamma: np.ndarray, xi: np.ndarray):
        T, D  = X.shape
        K, M  = self.n_states, self.n_gmm_components
        eye_D = np.eye(D) * 1e-6

        self.pi = gamma[0] / (gamma[0].sum() + 1e-12)
        xi_sum  = xi.sum(axis=0)
        self.A  = xi_sum / (xi_sum.sum(axis=1, keepdims=True) + 1e-12)

        gmm_g = self._gmm_posteriors(X, gamma)
        for s in range(K):
            gs = gamma[:, s].sum()
            if gs < 1e-12:
                continue
            for k in range(M):
                w    = gamma[:, s] * gmm_g[:, s, k]
                wsum = w.sum()
                if wsum < 1e-12:
                    continue
                self.gmm_weights[s, k] = wsum / gs
                mu                     = (w[:, None] * X).sum(axis=0) / wsum
                self.gmm_means[s, k]   = mu
                diff                   = X - mu
                self.gmm_covars[s, k]  = (
                    np.einsum('t,td,te->de', w, diff, diff) / wsum + eye_D)
            ws = self.gmm_weights[s].sum()
            if ws > 1e-12:
                self.gmm_weights[s] /= ws
        self._update_cholesky()

    # ------------------------------------------------------------------
    def _stationary_from_A(
        self,
        A,
        rel_tol:  float = 1e-8,
        abs_tol:  float = 1e-10,
        max_iter: int   = 2000,
        eps:      float = 1e-12,
        damping:  float = 0.0,
    ) -> np.ndarray:
        """
        Compute the stationary distribution of a transition matrix by
        power iteration with optional Laplacian damping.
        """
        A  = np.asarray(A, dtype=float)
        rs = A.sum(axis=1, keepdims=True)
        rs[rs == 0] = 1.0
        A  = A / rs
        K  = A.shape[0]
        if damping > 0.0:
            J = np.ones((K, K), dtype=float) / K
            A = (1.0 - damping) * A + damping * J
        pi = np.ones(K, dtype=float) / K
        for _ in range(max_iter):
            pi_new = pi @ A
            if (np.linalg.norm(pi_new - pi, 1)
                    <= rel_tol * np.linalg.norm(pi, 1) + abs_tol):
                pi = pi_new
                break
            pi = pi_new
        pi = np.maximum(pi, 0.0)
        s  = pi.sum()
        return pi / (s if s > 0 else (1.0 + eps))

    # ------------------------------------------------------------------
    def _stationary_flow_matrix(self, A: np.ndarray):
        """Return (flow_matrix, stationary_pi) where flow[i,j] = pi[i] * A[i,j]."""
        pi_s    = self._stationary_from_A(A)
        A_flow  = pi_s[:, None] * A
        return A_flow, pi_s

    # ------------------------------------------------------------------
    def _sequence_log_likelihood(self, X: np.ndarray) -> float:
        _, log_sc = self._forward(self._log_emission(X))
        return float(log_sc.sum())

    # ------------------------------------------------------------------
    def _snapshot_params(self) -> dict:
        return dict(
            pi=self.pi.copy(), A=self.A.copy(),
            weights=self.gmm_weights.copy(),
            means=self.gmm_means.copy(),
            covs=self.gmm_covars.copy(),
        )

    def _restore_params(self, snap: dict):
        self.pi          = snap['pi']
        self.A           = snap['A']
        self.gmm_weights = snap['weights']
        self.gmm_means   = snap['means']
        self.gmm_covars  = snap['covs']
        self._update_cholesky()

    # ------------------------------------------------------------------
    def fit(
        self,
        X_train:  np.ndarray,
        X_val:    np.ndarray,
        verbose:  bool = False,
    ):
        """
        Baum-Welch EM with three convergence criteria (first one met stops training):

            1. log_likelihood   : |train_ll(t) - train_ll(t-1)| < tol_ll
            2. stationary_pi    : ||pi_stat(t) - pi_stat(t-1)||_1 < tol_pi
            3. max_iterations   : iteration count reaches max_iter

        Parameters saved at the best validation log-likelihood iteration.
        """
        X_train = np.asarray(X_train, dtype=np.float64)
        X_val   = np.asarray(X_val,   dtype=np.float64)
        self._initialize_parameters(X_train)

        prev_train_ll = -np.inf
        prev_stat_pi  = np.ones(self.n_states) / self.n_states

        for iteration in range(self.max_iter):
            try:
                log_em        = self._log_emission(X_train)
                log_a, log_sc = self._forward(log_em)
                log_b         = self._backward(log_em, log_sc)
                gamma, xi     = self._e_step(log_a, log_b, log_em)

                train_ll = float(log_sc.sum())
                self.train_log_likelihoods.append(train_ll)
                self._m_step(X_train, gamma, xi)

                val_ll = self._sequence_log_likelihood(X_val)
                self.val_log_likelihoods.append(val_ll)

                if val_ll > self.best_val_ll:
                    self.best_val_ll       = val_ll
                    self.best_iteration    = iteration
                    self._best_params_snap = self._snapshot_params()

                # ----------------------------------------------------------
                # Convergence checks (OR logic — first met wins)
                # ----------------------------------------------------------
                curr_stat_pi = self._stationary_from_A(self.A)
                ll_delta     = abs(train_ll - prev_train_ll)
                pi_delta     = float(np.linalg.norm(curr_stat_pi - prev_stat_pi, 1))

                triggered = None
                if ll_delta < self.tol_ll:
                    triggered = 'log_likelihood'
                elif pi_delta < self.tol_pi:
                    triggered = 'stationary_pi'

                self.convergence_flags.append(
                    triggered if triggered is not None else 'running'
                )

                if verbose:
                    print(
                        f"    iter {iteration:3d}: "
                        f"train_ll={train_ll:.4f}  "
                        f"val_ll={val_ll:.4f}  "
                        f"ll_delta={ll_delta:.6f}  "
                        f"pi_delta={pi_delta:.6f}  "
                        f"flag={triggered or 'running'}"
                    )

                prev_train_ll = train_ll
                prev_stat_pi  = curr_stat_pi

                if triggered is not None:
                    self.converged             = True
                    self.convergence_criterion = triggered
                    self.final_iteration       = iteration
                    break

            except Exception as exc:
                if verbose:
                    print(f"    [ERROR] iter {iteration}: {exc}")
                self.convergence_flags.append('error')
                break

        if not self.converged:
            self.convergence_criterion = 'max_iterations'
            self.final_iteration       = len(self.train_log_likelihoods) - 1

        if self._best_params_snap is not None:
            self._restore_params(self._best_params_snap)

        try:
            self.stationary_pi = self._stationary_from_A(self.A)
        except Exception:
            self.stationary_pi = None

        return self

    # ------------------------------------------------------------------
    def get_parameters_dict(self) -> dict:
        params = {}
        for i in range(self.n_states):
            params[f'pi_{i}'] = float(self.pi[i])
        for i in range(self.n_states):
            for j in range(self.n_states):
                params[f'A_{i}{j}'] = float(self.A[i, j])
        for s in range(self.n_states):
            for k in range(self.n_gmm_components):
                params[f'gmm_weight_{s}_{k}'] = float(self.gmm_weights[s, k])
                for f in range(self.n_features):
                    params[f'gmm_mean_{s}_{k}_f{f}'] = float(
                        self.gmm_means[s, k, f])
                cov = self.gmm_covars[s, k]
                for fi in range(self.n_features):
                    for fj in range(fi, self.n_features):
                        params[f'gmm_cov_{s}_{k}_f{fi}f{fj}'] = float(
                            cov[fi, fj])
        params['final_train_log_likelihood'] = (
            float(self.train_log_likelihoods[-1])
            if self.train_log_likelihoods else float('nan'))
        params['best_val_log_likelihood']  = float(self.best_val_ll)
        params['best_iteration']           = int(self.best_iteration)
        params['converged']                = bool(self.converged)
        params['final_iteration']          = int(self.final_iteration)
        params['convergence_criterion']    = str(self.convergence_criterion)
        params['n_states']                 = int(self.n_states)
        params['n_gmm_components']         = int(self.n_gmm_components)
        params['n_features']               = int(self.n_features)
        if self.stationary_pi is not None:
            for i, p in enumerate(self.stationary_pi):
                params[f'stationary_pi_{i}'] = float(p)
        return params


# =============================================================================
# Checkpoint manager
# =============================================================================

class CheckpointManager:
    def __init__(self, output_dir: str):
        self.output_dir    = output_dir
        self.ckpt_dir      = os.path.join(output_dir, 'checkpoints')
        self.progress_path = os.path.join(output_dir, 'progress.json')
        os.makedirs(self.ckpt_dir, exist_ok=True)
        self._progress = self._load_progress()

    def _load_progress(self) -> dict:
        if os.path.exists(self.progress_path):
            with open(self.progress_path, 'r') as f:
                return json.load(f)
        return {
            'completed':   [],
            'failed':      [],
            'started':     datetime.now().isoformat(),
            'last_update': None,
        }

    def _save_progress(self):
        self._progress['last_update'] = datetime.now().isoformat()
        with open(self.progress_path, 'w') as f:
            json.dump(self._progress, f, indent=2)

    def is_done(self, label: str) -> bool:
        return label in self._progress['completed']

    def is_failed(self, label: str) -> bool:
        return any(e['label'] == label for e in self._progress['failed'])

    def save(self, label: str, row: dict, curve: dict, hmm_obj):
        ckpt_path = os.path.join(self.ckpt_dir, f"{label}.pkl")
        with open(ckpt_path, 'wb') as f:
            pickle.dump({
                'label':     label,
                'row':       row,
                'curve':     curve,
                'hmm':       hmm_obj,
                'timestamp': datetime.now().isoformat(),
            }, f)
        if label not in self._progress['completed']:
            self._progress['completed'].append(label)
        self._save_progress()

    def mark_failed(self, label: str, error: str):
        self._progress['failed'].append({
            'label':     label,
            'error':     str(error)[:300],
            'timestamp': datetime.now().isoformat(),
        })
        self._save_progress()

    def load_all(self) -> tuple:
        all_rows, all_curves = [], []
        for label in self._progress['completed']:
            ckpt_path = os.path.join(self.ckpt_dir, f"{label}.pkl")
            if not os.path.exists(ckpt_path):
                print(f"  [WARNING] Missing checkpoint file: {label}")
                continue
            try:
                with open(ckpt_path, 'rb') as f:
                    p = pickle.load(f)
                all_rows.append(p['row'])
                all_curves.append(p['curve'])
            except Exception as exc:
                print(f"  [WARNING] Could not load {label}: {exc}")
        return all_rows, all_curves

    def print_status(self, total: int):
        n_done   = len(self._progress['completed'])
        n_failed = len(self._progress['failed'])
        print(f"\n  --- checkpoint status ---")
        print(f"  completed : {n_done}/{total}")
        print(f"  failed    : {n_failed}")
        print(f"  remaining : {max(0, total - n_done - n_failed)}")
        print(f"  last save : {self._progress.get('last_update', 'never')}")
        if self._progress['failed']:
            for e in self._progress['failed']:
                print(f"    {e['label']} - {e['error'][:80]}")


# =============================================================================
# Collect results
# =============================================================================

def collect_results(
    output_dir:       str,
    n_states:         int,
    n_gmm_components: int,
    max_iter:         int,
    train_ratio:      float,
    tol:              float,
):
    csv_path  = os.path.join(output_dir, 'hmm_results.csv')
    json_path = os.path.join(output_dir, 'hmm_learning_curves.json')
    manager   = CheckpointManager(output_dir)
    all_rows, all_curves = manager.load_all()
    if not all_rows:
        print("  [collect] No completed checkpoints found.")
        return None, None
    df = pd.DataFrame(all_rows)
    df.to_csv(csv_path, index=False)
    print(f"\n  CSV  -> {csv_path}  ({len(df)} rows, {len(df.columns)} cols)")
    print(f"     CONTROL={len(df[df['subject_type']=='CONTROL'])}  "
          f"ADHD={len(df[df['subject_type']=='ADHD'])}")
    json_payload = {
        'metadata': {
            'created':         datetime.now().isoformat(),
            'n_subjects':      len(all_curves),
            'n_states':        n_states,
            'n_gmm_components':n_gmm_components,
            'max_iter':        max_iter,
            'train_ratio':     train_ratio,
            'tol':             tol,
            'description': (
                "Learning curves per subject. "
                "best_iteration = index of highest val LL; "
                "saved parameters correspond to that iteration. "
                "convergence_criterion: one of "
                "log_likelihood | stationary_pi | max_iterations."
            ),
        },
        'curves': all_curves,
    }
    with open(json_path, 'w') as f:
        json.dump(json_payload, f, indent=2)
    print(f"  JSON -> {json_path}  ({len(all_curves)} curves)")
    return df, json_payload


# =============================================================================
# Combined summary across multiple configurations
# =============================================================================

def build_combined_summary(
    base_output_dir: str,
    configs:         list,
) -> pd.DataFrame:
    """
    Read hmm_results.csv from every NxGy directory in `configs` and produce:

        combined_summary.csv        -- one row per (subject, config)
        best_config_per_subject.csv -- one row per subject with
                                       best config by raw and normalised val LL
    """
    all_dfs = []
    print("\n  Building combined summary ...")

    for s, g in configs:
        tag      = _config_tag(s, g)
        csv_path = os.path.join(
            _config_output_dir(base_output_dir, s, g), 'hmm_results.csv'
        )
        if not os.path.exists(csv_path):
            print(f"    [MISSING] {tag}: {csv_path}")
            continue
        df = pd.read_csv(csv_path)
        df['config_tag']             = tag
        df['n_states']               = s
        df['n_gmm_components']       = g
        df['best_val_ll_per_sample'] = (
            df['best_val_log_likelihood'] / df['val_samples']
        )
        all_dfs.append(df)
        print(f"    {tag}: {len(df)} rows")

    if not all_dfs:
        print("  No CSV files found.")
        return None

    combined  = pd.concat(all_dfs, ignore_index=True)

    meta_cols = [
        'config_tag', 'n_states', 'n_gmm_components',
        'subject_type', 'subject_id', 'label',
        'n_samples', 'n_channels', 'train_samples', 'val_samples',
        'best_val_log_likelihood', 'best_val_ll_per_sample',
        'best_iteration', 'final_iteration', 'convergence_criterion',
        'converged', 'training_time_sec', 'random_seed',
    ]
    param_cols = [c for c in combined.columns if c not in meta_cols]
    combined   = combined[
        [c for c in meta_cols if c in combined.columns] + param_cols
    ]

    combined_path = os.path.join(base_output_dir, 'combined_summary.csv')
    combined.to_csv(combined_path, index=False)
    print(f"\n  Combined summary -> {combined_path}")
    print(f"    Rows    : {len(combined)}")
    print(f"    Configs : {combined['config_tag'].nunique()}/{len(configs)}")
    print(f"    Subjects: {combined['label'].nunique()}")

    # ------------------------------------------------------------------
    # Convergence criterion distribution
    # ------------------------------------------------------------------
    print("\n  Convergence criterion distribution:")
    print(combined['convergence_criterion'].value_counts().to_string())

    # ------------------------------------------------------------------
    # Best config per subject
    # ------------------------------------------------------------------
    records_best = []
    for label, grp in combined.groupby('label'):
        meta      = grp.iloc[0][['subject_type', 'subject_id', 'label']]
        idx_raw   = grp['best_val_log_likelihood'].idxmax()
        idx_norm  = grp['best_val_ll_per_sample'].idxmax()
        best_raw  = grp.loc[idx_raw]
        best_norm = grp.loc[idx_norm]
        records_best.append({
            'label':            label,
            'subject_type':     meta['subject_type'],
            'subject_id':       meta['subject_id'],
            'best_config_raw':  best_raw['config_tag'],
            'best_val_ll_raw':  best_raw['best_val_log_likelihood'],
            'best_iter_raw':    best_raw['best_iteration'],
            'best_config_norm': best_norm['config_tag'],
            'best_val_ll_norm': best_norm['best_val_ll_per_sample'],
            'best_iter_norm':   best_norm['best_iteration'],
            'raw_norm_agree':   (
                best_raw['config_tag'] == best_norm['config_tag']
            ),
        })

    best_df   = pd.DataFrame(records_best)
    best_path = os.path.join(base_output_dir, 'best_config_per_subject.csv')
    best_df.to_csv(best_path, index=False)
    print(f"  Best config    -> {best_path}")
    print(f"\n  Best config distribution (normalised LL / sample):")
    print(best_df['best_config_norm'].value_counts().to_string())
    print(f"\n  Raw vs normalised agreement: "
          f"{best_df['raw_norm_agree'].sum()}/{len(best_df)} subjects")

    return combined


# =============================================================================
# Entry point
# =============================================================================

if __name__ == "__main__":
    # !pip install numba --quiet

    # -------------------------------------------------------------------------
    # Paths
    # -------------------------------------------------------------------------
    CONTROL_PREP_DIR = ".../preprocessed/Control_prep"
    ADHD_PREP_DIR    = ".../preprocessed/ADHD_prep"
    BASE_DIR         = ".../hmm_results"

    # -------------------------------------------------------------------------
    # Configuration selection
    #
    #   CONFIGS_TO_RUN = 'all'              -> all 40 combinations (N1G1..N8G5)
    #   CONFIGS_TO_RUN = [(3, 3), (5, 2)]   -> specific (n_states, n_gmm) pairs
    # -------------------------------------------------------------------------
    CONFIGS_TO_RUN = [(4, 4), (5, 5)]

    # -------------------------------------------------------------------------
    # Training hyperparameters
    # -------------------------------------------------------------------------
    MAX_HOURS      = 12.0 # ----------------------------------------------------------------CHANGE THE HOURS 
    BUFFER_MINUTES = 30.0
    TRAIN_RATIO    = 0.70
    VERBOSE        = True

    # Convergence tolerance and scale factors
    # Effective thresholds:
    #   tol_ll = TOL * TOL_SCALE_LL   (log-likelihood delta)
    #   tol_pi = TOL * TOL_SCALE_PI   (stationary pi L1 norm delta)
    # max_iter is handled separately per config via _max_iter_for_states().
    TOL           = 1.0
    TOL_SCALE_LL  = 1.0      # tol_ll = 1.0  (LL scale ~ tens to thousands)
    TOL_SCALE_PI  = 1e-3     # tol_pi = 0.001 (pi lives in [0, 2])

    # -------------------------------------------------------------------------
    # Resolve and validate configuration list
    # -------------------------------------------------------------------------
    configs = resolve_configs(CONFIGS_TO_RUN)
    existing = discover_existing_configs(BASE_DIR)

    print("=" * 70)
    print("  MULTI-CONFIGURATION HMM TRAINING")
    print("  Dataset: ADHD / Control EEG")
    print("=" * 70)
    print(f"  Configurations requested : {len(configs)}")
    print(f"  Configurations on disk   : {len(existing)}")
    print(f"  TOL={TOL}  tol_ll={TOL * TOL_SCALE_LL}  "
          f"tol_pi={TOL * TOL_SCALE_PI}")

    for s, g in configs:
        tag      = _config_tag(s, g)
        max_iter = _max_iter_for_states(s)
        out_d    = _config_output_dir(BASE_DIR, s, g)
        done     = 0
        prog_f   = os.path.join(out_d, 'progress.json')
        if os.path.exists(prog_f):
            try:
                with open(prog_f) as f:
                    done = len(json.load(f).get('completed', []))
            except Exception:
                pass
        status = "[EXISTS]" if (s, g) in existing else "[NEW]"
        print(f"    {tag}  max_iter={max_iter:3d}  "
              f"completed={done}/121  {status}")
    print("=" * 70)

    timer   = TimeManager(max_hours=MAX_HOURS)
    records = discover_files(CONTROL_PREP_DIR, ADHD_PREP_DIR)
    n_subjects = len(records)
    print(f"\n  Files discovered: {n_subjects}")

    # -------------------------------------------------------------------------
    # Outer loop: configurations
    # -------------------------------------------------------------------------
    for cfg_idx, (n_states, n_gmm) in enumerate(configs):
        tag      = _config_tag(n_states, n_gmm)
        max_iter = _max_iter_for_states(n_states)
        out_dir  = _config_output_dir(BASE_DIR, n_states, n_gmm)

        if timer.should_stop(buffer_minutes=BUFFER_MINUTES):
            print(f"\n  [TIME LIMIT] Before {tag}. "
                  f"Elapsed={timer.format_elapsed()}. Re-run to continue.")
            break

        print(f"\n{'='*70}")
        print(f"  CONFIG [{cfg_idx+1}/{len(configs)}]  {tag}  "
              f"(n_states={n_states}, n_gmm={n_gmm}, max_iter={max_iter})")
        print(f"  Elapsed: {timer.format_elapsed()}  "
              f"Remaining: {timer.remaining_hours():.2f}h")
        print(f"{'='*70}")

        os.makedirs(out_dir, exist_ok=True)
        manager = CheckpointManager(out_dir)
        manager.print_status(total=n_subjects)

        for idx, (filepath, subject_type, subject_id) in enumerate(records):
            label = f"{subject_type}_{subject_id}"

            if timer.should_stop(buffer_minutes=BUFFER_MINUTES):
                n_done = len(manager._progress['completed'])
                print(f"\n  [TIME LIMIT] During {tag} after {n_done} subjects. "
                      f"Re-run to resume.")
                break

            if manager.is_done(label):
                print(f"  [{idx+1:3d}/{n_subjects}]  {tag}  {label}  [SKIP]")
                continue
            if manager.is_failed(label):
                print(f"  [{idx+1:3d}/{n_subjects}]  {tag}  {label}  [FAILED]")
                continue

            print(f"\n  [{idx+1:3d}/{n_subjects}]  {tag}  {label}")

            try:
                X, mat_label, mat_sid = load_preprocessed_mat(filepath)

                # Cross-check embedded metadata against directory-derived label.
                if mat_label is not None:
                    mat_type = mat_label.upper()
                    if mat_type != subject_type:
                        print(f"    [WARNING] Directory type '{subject_type}' "
                              f"does not match .mat label '{mat_label}'. "
                              f"Using directory-derived type.")
                if mat_sid is not None and mat_sid != subject_id:
                    print(f"    [WARNING] Filename subject_id '{subject_id}' "
                          f"does not match .mat subject_id '{mat_sid}'.")

                n_samples, n_ch = X.shape
                split           = int(np.floor(n_samples * TRAIN_RATIO))
                X_train, X_val  = X[:split], X[split:]
                print(f"    signal : {n_samples} x {n_ch}  |  "
                      f"train={len(X_train)}  val={len(X_val)}")

                seed = _make_seed(subject_type, subject_id)
                t0   = time.time()
                hmm  = MultivariateGaussianMixtureHMM(
                    n_states=n_states,
                    n_gmm_components=n_gmm,
                    max_iter=max_iter,
                    tol=TOL,
                    tol_scale_ll=TOL_SCALE_LL,
                    tol_scale_pi=TOL_SCALE_PI,
                    random_state=seed,
                )
                hmm.fit(X_train, X_val, verbose=VERBOSE)
                elapsed = time.time() - t0

                print(f"    done   : {elapsed:.1f}s  |  "
                      f"best_iter={hmm.best_iteration}  "
                      f"best_val_ll={hmm.best_val_ll:.4f}  "
                      f"criterion={hmm.convergence_criterion}")

                row = {
                    'subject_type':      subject_type,
                    'subject_id':        subject_id,
                    'label':             label,
                    'n_samples':         n_samples,
                    'n_channels':        n_ch,
                    'train_samples':     len(X_train),
                    'val_samples':       len(X_val),
                    'training_time_sec': round(elapsed, 2),
                    'random_seed':       seed,
                }
                row.update(hmm.get_parameters_dict())

                curve = {
                    'label':              label,
                    'subject_type':       subject_type,
                    'subject_id':         subject_id,
                    'config_tag':         tag,
                    'n_states':           n_states,
                    'n_gmm_components':   n_gmm,
                    'max_iter':           max_iter,
                    'train_ratio':        TRAIN_RATIO,
                    'n_train_samples':    int(len(X_train)),
                    'n_val_samples':      int(len(X_val)),
                    'random_seed':        int(seed),
                    'train_log_likelihoods': [
                        round(v, 6) for v in hmm.train_log_likelihoods],
                    'val_log_likelihoods': [
                        round(v, 6) for v in hmm.val_log_likelihoods],
                    'convergence_flags':       hmm.convergence_flags,
                    'best_iteration':          int(hmm.best_iteration),
                    'best_val_log_likelihood': round(hmm.best_val_ll, 6),
                    'converged':               bool(hmm.converged),
                    'convergence_criterion':   str(hmm.convergence_criterion),
                    'iterations_run':          len(hmm.train_log_likelihoods),
                    'training_time_sec':       round(elapsed, 2),
                }

                manager.save(label, row, curve, hmm)
                del hmm
                gc.collect()

                n_done = len(manager._progress['completed'])
                if n_done % 5 == 0:
                    manager.print_status(total=n_subjects)
                    timer.print_status(completed=n_done, total=n_subjects)

            except Exception as exc:
                print(f"    [ERROR] {tag} {label}: {exc}")
                manager.mark_failed(label, str(exc))
                continue

        # Flush per-config CSV + JSON
        collect_results(
            output_dir=out_dir,
            n_states=n_states,
            n_gmm_components=n_gmm,
            max_iter=max_iter,
            train_ratio=TRAIN_RATIO,
            tol=TOL,
        )
        manager.print_status(total=n_subjects)

    # -------------------------------------------------------------------------
    # Combined summary across all trained configurations
    # -------------------------------------------------------------------------
    print("\n" + "=" * 70)
    build_combined_summary(base_output_dir=BASE_DIR, configs=configs)

    n_total_done = sum(
        len(
            json.load(open(os.path.join(
                _config_output_dir(BASE_DIR, s, g), 'progress.json'
            ))).get('completed', [])
        )
        for s, g in configs
        if os.path.exists(os.path.join(
            _config_output_dir(BASE_DIR, s, g), 'progress.json'))
    )
    timer.print_status(
        completed=n_total_done,
        total=len(configs) * n_subjects,
    )
    print("\n" + "=" * 70)
    print("  TRAINING RUN COMPLETE")
    print("=" * 70)